# Watermark comparison on Colab

Qwen 1.5B, all 164 HumanEval problems, and KGW strengths 1, 2, and 3. Cloud results are separate from laptop runs. Results pending execution.

## Environment

Pinned dependencies in a separate Python environment. GPU precision is bfloat16 where supported, otherwise float32. Drive holds resumable checkpoints.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys, zipfile
from IPython.display import Image, Markdown, display

USE_VSCODE = True  # False enables Colab browser upload/download dialogs.
USE_DRIVE = True  # False uses temporary storage; download results before leaving.
READY = False
if not Path('/content').is_dir():
    raise RuntimeError('This notebook uses a Google Colab GPU runtime.')
from google.colab import drive, files
if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU using Select Kernel > Colab > New Colab Server, then rerun.')
DRIVE_ROOT = None
if USE_DRIVE:
    try:
        drive.mount('/content/drive')
    except Exception as exc:
        raise RuntimeError('Drive authorization failed. In VS Code, run Colab: Mount Google Drive to Server, '
                           'execute its added cell and retry; or set USE_DRIVE=False for temporary storage.') from exc
    DRIVE_ROOT = Path('/content/drive/MyDrive/CodeGuardLLM/colab')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

PROJECT = Path('/content/CodeGuardLLM')
BUNDLE = Path('/content/codeguard-colab.zip')
if not (PROJECT / 'src/codeguard/cloud.py').is_file():
    saved_bundle = DRIVE_ROOT / BUNDLE.name if DRIVE_ROOT else None
    if not BUNDLE.is_file() and saved_bundle and saved_bundle.is_file():
        shutil.copyfile(saved_bundle, BUNDLE)
    if not BUNDLE.is_file():
        if USE_VSCODE:
            raise FileNotFoundError('In VS Code Explorer, right-click .tmp/codeguard-colab.zip > Upload to Colab. '
                                    'Wait for upload to finish, then rerun this cell.')
        print('Select codeguard-colab.zip from your laptop.')
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise RuntimeError('Select only the project ZIP.')
        BUNDLE.write_bytes(next(iter(uploaded.values())))
        del uploaded
    with zipfile.ZipFile(BUNDLE) as archive:
        for item in archive.infolist():
            target = (PROJECT.parent / item.filename).resolve()
            if (not target.is_relative_to(PROJECT) or '\\' in item.filename
                    or (item.external_attr >> 16) & 0o170000 == 0o120000):
                raise ValueError('Invalid project archive path')
        if 'CodeGuardLLM/src/codeguard/cloud.py' not in archive.namelist():
            raise ValueError('Use the newly prepared codeguard-colab.zip.')
        archive.extractall(PROJECT.parent)
    if saved_bundle:
        shutil.copyfile(BUNDLE, saved_bundle)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / 'src'))
from codeguard.cloud import Checkpoint
ENV = PROJECT / '.venv-colab'
PYTHON = str(ENV / 'bin/python')
CHECKPOINT = None
print('Project:', PROJECT)

In [ ]:
from collections import deque
from datetime import datetime, timezone
import os, subprocess

LOG_DIR = PROJECT / '.tmp/notebook-logs'
LOG_DIR.mkdir(parents=True, exist_ok=True)

def command(args, label='command'):
    log = LOG_DIR / (label + '-' + datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f') + '.log')
    tail = deque(maxlen=30)
    print(f'Running {label}. Log: {log}', flush=True)
    with log.open('w', encoding='utf-8') as handle:
        with subprocess.Popen(
            list(map(str, args)), cwd=PROJECT, stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT, text=True, encoding='utf-8', errors='replace',
            bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
        ) as process:
            try:
                for line in process.stdout:
                    print(line, end='', flush=True)
                    handle.write(line)
                    handle.flush()
                    tail.append(line)
                    if CHECKPOINT is not None:
                        CHECKPOINT.save()
                status = process.wait()
            except BaseException:
                process.terminate()
                try:
                    process.wait(timeout=10)
                except subprocess.TimeoutExpired:
                    process.kill()
                    process.wait()
                raise
            finally:
                if CHECKPOINT is not None:
                    CHECKPOINT.save(force=True)
    if status:
        detail = ''.join(tail)
        raise RuntimeError(f'{label} failed (exit {status}). Full log: {log}\n\n{detail}')

def run(*args, label='codeguard'):
    command([PYTHON, '-u', *args], label=label)

In [ ]:
READY = False
# virtualenv supplies pip even when Colab's Python has no ensurepip.
if not Path(PYTHON).is_file():
    command([sys.executable, '-m', 'pip', 'install', 'virtualenv'], label='environment-tool')
    command([sys.executable, '-m', 'virtualenv', str(ENV)], label='environment')
command([sys.executable, '-m', 'pip', '--python', PYTHON, 'install', '--upgrade', 'pip'], label='pip')
run('-m', 'pip', 'install', 'torch==2.8.0', '--index-url', 'https://download.pytorch.org/whl/cu126', label='pytorch')
run('-m', 'pip', 'install', '-e', '.[ml,data,analysis]', label='dependencies')
run('scripts/bootstrap_markllm.py', label='markllm')

run('-c', """
import json
from pathlib import Path
import torch, yaml
from codeguard.backend import HFBackend
from codeguard.cloud import select_dtype
from codeguard.config import load_config
from codeguard.storage import digest
assert torch.cuda.is_available(), 'Select a Colab GPU runtime before continuing'
config = load_config('configs/gpu.yaml')
config['model']['dtype'] = select_dtype(torch.cuda.get_device_capability()[0], torch.cuda.is_bf16_supported())
x = torch.ones((16,16), device='cuda', dtype=getattr(torch, config['model']['dtype']))
assert torch.isfinite(x @ x).all().item(), 'GPU calculation failed'
print('GPU:', torch.cuda.get_device_name(), '| Precision:', config['model']['dtype'])
metadata = HFBackend(config, load_model=False).metadata
Path('.tmp/colab.yaml').write_text(yaml.safe_dump(config), encoding='utf-8')
Path('.tmp/colab-runtime.json').write_text(json.dumps({'id': digest(metadata)[:16], 'backend': metadata}), encoding='utf-8')
""", label='gpu-and-detector')
CONFIG = PROJECT / '.tmp/colab.yaml'
runtime = json.loads((PROJECT / '.tmp/colab-runtime.json').read_text())
RUN_DIR = PROJECT / 'runs/colab' / runtime['id'] / 'watermark_comparison'
backup = DRIVE_ROOT / runtime['id'] / 'results.zip' if DRIVE_ROOT else None
CHECKPOINT = Checkpoint(RUN_DIR, backup, PROJECT / '.tmp/checkpoints')
CHECKPOINT.restore()
RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copyfile(CONFIG, RUN_DIR / 'colab.yaml')
(RUN_DIR / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
CHECKPOINT.save(force=True)
run('-m', 'codeguard', 'validate', '--config', CONFIG, '--check-markllm', label='validate')
READY = True
print('Results:', RUN_DIR)
print('Drive checkpoint:', backup if backup else 'disabled: temporary storage only')

## Samples

The experiment contains 984 completions: 164 problems × 2 conditions × 3 runs. Existing samples are reused when resuming.

In [ ]:
if not globals().get('READY', False):
    raise RuntimeError('Setup has not completed.')
run('scripts/compare_watermarks.py', 'generate', '--config', CONFIG, '--output', RUN_DIR, label='watermark_comparison-generation')


## Reference controls

The 164 HumanEval reference solutions form the human-code control group. Scores are shared across strengths; no model weights or code execution are needed.

In [ ]:
run('scripts/compare_watermarks.py', 'controls', '--config', CONFIG, '--output', RUN_DIR, label='reference-controls')


## Correctness

Pass@1 remains pending here. Exported solutions are tested in Docker on the laptop, without model inference.

## Results

Tables and plots below are computed from saved samples.

In [ ]:
run('scripts/compare_watermarks.py', 'analyze', '--config', CONFIG, '--output', RUN_DIR, label='core-analysis')
summary = json.loads((RUN_DIR / 'analysis/core_summary.json').read_text(encoding='utf-8'))
def number(value):
    return 'pending / unavailable' if value is None else f'{value:.3f}'
table = ['| Strength | AUROC vs clean | AUROC vs reference | Clean Pass@1 | Watermarked Pass@1 | Change |',
         '|---|---|---|---|---|---|']
for row in summary['comparison']:
    table.append(f"| {row['delta']:g} | {number(row['auroc_vs_clean'])} | {number(row['auroc_vs_human_reference'])} | "
                 f"{number(row['clean_pass_at_1'])} | {number(row['watermarked_pass_at_1'])} | {number(row['pass_at_1_change'])} |")
display(Markdown('\n'.join(table)))
display(Image(filename=str(RUN_DIR / 'analysis/controls_detection.png')))
display(Image(filename=str(RUN_DIR / 'analysis/tradeoff.png')))
print('Report status:', summary['status'])
print('Written report:', RUN_DIR / 'analysis/report.md')


## Saved experiment

The archive contains samples, manifests, tables, and plots.

In [ ]:
# This cell also exports completed samples after an interrupted generation.
if not any(RUN_DIR.glob('*/generations.jsonl')):
    print('No samples saved yet.')
else:
    CHECKPOINT.save(force=True)
    EXPORT = shutil.make_archive(str(PROJECT.parent / 'codeguard-colab-results'), 'zip', RUN_DIR.parent, RUN_DIR.name)
    print('Results archive:', EXPORT)
    if USE_VSCODE:
        print('In the VS Code Colab sidebar > Contents, refresh, then right-click codeguard-colab-results.zip > Download...')
    else:
        files.download(EXPORT)